In [1]:
# Lists
# An ordered collection in square brackets:

lures = ["Spinner", "Jig", "Crankbait"]

print(lures[0])     # Spinner (indexing starts at 0)
print(lures[-1])    # Crankbait (negative counts from the end)
print(len(lures))   # 3 (like COUNT(*))


# Modifying a list:
lures.append("Worm")        # add to the end
lures.remove("Jig")         # remove by value
lures[0] = "Popper"         # replace an item
print(lures)                # ['Popper', 'Crankbait', 'Worm']

# Slicing grabs a range, like TOP or OFFSET/FETCH:
nums = [10, 20, 30, 40, 50]
print(nums[0:3])   # [10, 20, 30] (end is excluded)
print(nums[:2])    # [10, 20] (first 2)
print(nums[-2:])   # [40, 50] (last 2)

# Useful built-ins, your aggregate functions:
prices = [12.99, 4.50, 25.00, 8.75]
print(sum(prices))      # SUM
print(min(prices))      # MIN
print(max(prices))      # MAX
print(sorted(prices))   # ORDER BY (returns a new list)
print(sorted(prices, reverse=True))   # ORDER BY ... DESC

Spinner
Crankbait
3
['Popper', 'Crankbait', 'Worm']
[10, 20, 30]
[10, 20]
[40, 50]
51.24
4.5
25.0
[4.5, 8.75, 12.99, 25.0]
[25.0, 12.99, 8.75, 4.5]


In [3]:
# Dictionaries
# Key–value pairs in curly braces, like one row with named columns:

order = {
    "order_id": 101,
    "customer": "Smith",
    "amount": 45.00
}

print(order["customer"])     # Smith
order["status"] = "Shipped"  # add a new key (like adding a column)
order["amount"] = 50.00      # update a value

# Accessing a key that doesn’t exist raises an error. .get() is safer and returns None instead, similar to a NULL:
print(order.get("discount"))        # None
print(order.get("discount", 0))     # 0 (default value, like COALESCE)


# Looping over a dictionary:
for key, value in order.items():
    print(f"{key}: {value}")

Smith
None
0
order_id: 101
customer: Smith
amount: 50.0
status: Shipped


In [4]:
# A list of dictionaries = a table
# This is the shape you’ll get from APIs, JSON files, and many database drivers:

orders = [
    {"order_id": 101, "customer": "Smith", "amount": 45.00},
    {"order_id": 102, "customer": "Jones", "amount": 120.50},
    {"order_id": 103, "customer": "Smith", "amount": 8.99},
]

# SELECT customer, amount FROM orders WHERE amount > 40
for o in orders:
    if o["amount"] > 40:
        print(o["customer"], o["amount"])

Smith 45.0
Jones 120.5


In [5]:
# List comprehensions
# A compact one-line way to build a list. You’ll see these everywhere:

# SELECT amount FROM orders
amounts = [o["amount"] for o in orders]

# SELECT amount FROM orders WHERE customer = 'Smith'
smith_amounts = [o["amount"] for o in orders if o["customer"] == "Smith"]

print(amounts)         # [45.0, 120.5, 8.99]
print(smith_amounts)   # [45.0, 8.99]

# Read it as: give me o["amount"] for each o in orders, where the condition is true.

[45.0, 120.5, 8.99]
[45.0, 8.99]


In [6]:
# Two more (data structure types) you’ll run into

# Tuple: like a list, but can't be changed after creation
point = (33.75, -84.39)

# Set: unique values only, like SELECT DISTINCT
customers = {"Smith", "Jones", "Smith"}
print(customers)   # {'Smith', 'Jones'}

# DISTINCT from a list:
unique_customers = set(o["customer"] for o in orders)

{'Smith', 'Jones'}


In [23]:
# Exercise
# Using this data:

orders = [
    {"order_id": 101, "customer": "Smith", "amount": 45.00},
    {"order_id": 102, "customer": "Jones", "amount": 120.50},
    {"order_id": 103, "customer": "Smith", "amount": 8.99},
    {"order_id": 104, "customer": "Lee",   "amount": 250.00},
    {"order_id": 105, "customer": "Jones", "amount": 63.25},
]

# Print the number of orders.
print(len(orders))

# Use a list comprehension to get all amounts over 50.
large_amounts = [o["amount"] for o in orders if o["amount"] > 50]
print(large_amounts) 

# Print the distinct customers.
unique_customers = {o["customer"] for o in orders} # OR unique_customers = set(o["customer"] for o in orders), had warning
print(unique_customers) 

# Bonus (this one is a GROUP BY): build a dictionary of total amount per customer, so the result looks like {"Smith": 53.99, "Jones": 183.75, "Lee": 250.0}.

# Hint for the bonus: start with an empty dict totals = {}, loop through the orders, and use .get() with a default of 0.
totals = {}

# solution 1
for o in orders:
    customer = o["customer"]
    amount = o["amount"]
    totals[customer] = totals.get(customer, 0) + amount

print(totals)

# solution 2 (nicer print format)
for customer, total in totals.items():
    print(f"{customer}: ${total:.2f}")

# solution 3 (when using panadas)
# df.groupby("customer")["amount"].sum()

# same, but with total orders count instead of total orders amount
counts = {}

for o in orders:
    customer = o["customer"]
    counts[customer] = counts.get(customer, 0) + 1

print(counts)

# another solution (nicer print format)
for customer, counts in counts.items():
    print(f"{customer}: {counts}")



5
[120.5, 250.0, 63.25]
{'Lee', 'Smith', 'Jones'}
{'Smith': 53.99, 'Jones': 183.75, 'Lee': 250.0}
Smith: $53.99
Jones: $183.75
Lee: $250.00
{'Smith': 2, 'Jones': 2, 'Lee': 1}
Smith: 2
Jones: 2
Lee: 1


In [24]:
# GROUP BY customer: COUNT(*) and SUM(amount)
totals = {}
counts = {}

for o in orders:
    customer = o["customer"]
    totals[customer] = totals.get(customer, 0) + o["amount"]
    counts[customer] = counts.get(customer, 0) + 1

# Print a report, sorted by customer name (ORDER BY customer)
print(f"{'Customer':<10}{'Orders':>8}{'Total':>12}")
print("-" * 30)

for customer in sorted(totals):
    print(f"{customer:<10}{counts[customer]:>8}{totals[customer]:>12,.2f}")

# Grand total row
print("-" * 30)
print(f"{'ALL':<10}{len(orders):>8}{sum(totals.values()):>12,.2f}")

Customer    Orders       Total
------------------------------
Jones            2      183.75
Lee              1      250.00
Smith            2       53.99
------------------------------
ALL              5      487.74


In [26]:
# New formatting tricks in there:

# :<10 left-aligns in a 10-character space, and :>8 right-aligns in 8, so the columns line up.
# :,.2f adds thousands separators and two decimals (1,250.00).
# "-" * 30 repeats a character 30 times.
# sorted(totals) sorts the dictionary’s keys alphabetically.

# To sort by total amount, highest first (ORDER BY total DESC), swap the loop line for:


# for customer in sorted(totals, key=totals.get, reverse=True):

# The SQL equivalent of the whole thing is:
# SELECT customer, COUNT(*) AS orders, SUM(amount) AS total
# FROM orders
# GROUP BY customer
# ORDER BY customer;